In [1]:
%matplotlib inline 

import os, sys
import xarray as xr
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [75]:
def subset_mapping(ds, ids):
    """
    Subset a mapping dataset to selected polygon IDs.

    Parameters
    ----------
    ds : xr.Dataset
        Mapping dataset containing:
        - polyid on the polygon dimension
        - overlaps on the polygon dimension
        - weight on the data dimension
        - i_index and j_index on the data dimension
    ids : array-like
        Polygon IDs to retain.

    Returns
    -------
    xr.Dataset
    """
    
    # get dimension names                                                                                                                
    polyDim=ds.polyid.dims                                                                                                               
    dataDim=ds.weight.dims                                                                                                               
                                                                                                                                       
    # split reach dim data and hru dim data                                                                                              
    ds_poly = ds.drop_dims(dataDim)                                                                                                      
    ds_data = ds.drop_dims(polyDim)                                                                                                      
                                                                                                                                       
    # Check data dimension size and sum of overlapping elements (expected to be the same)                                                
    # if target polygon does not have any overlapped source polygons,                                                                    
    # grid2poly.py output skip in variables with data dimension, but poly2poly put missing value (zero) in varialbes with data dimension
    dim_data_size = ds_mapping.sizes["data"]                                                                                                      
    sum_overlaps  = ds.overlaps.values.sum()                                                                                             
    if dim_data_size != sum_overlaps:                                                                                                    
      print('WARNING: data dimension size (%d) is not equal to sum of overlapping elements (%d)'%(dim_data_size, sum_overlaps))        
      if 'overlapPolyId' in ds_data.variables:                                                                                         
          varname_intersector = 'overlapPolyId'                                                                                        
      elif 'intersector' in ds_data.variables:                                                                                         
          varname_intersector = 'intersector'                                                                                          
      ds_data = ds_data.where(ds[varname_intersector]!=0, drop=True)                                                                   
      print(f'         remove element corresponding to {varname_intersector}==0')                                                      
                                                                                                                                       
    # subset poly dimension variables                                                                                                    
    idx_mask = ds_poly.polyid.isin(ids)                                                                                                  
    ds_poly_sub = ds_poly.where(idx_mask, drop=True)                                                                                     
                                                                                                                                       
    # subset data dimension variables                                                                                                    
    idx_end   = np.cumsum(ds.overlaps.values)                                                                                            
    idx_start = idx_end - ds.overlaps.values                                                                                             
                                                                                                                                       
    idx_start_sub = idx_start[np.where(idx_mask)]                                                                                        
    idx_end_sub   = idx_end[np.where(idx_mask)]                                                                                          
                                                                                                                                       
    idx_data_sub =[]                                                                                                                     
    for ix1,ix2 in zip(idx_start_sub, idx_end_sub):                                                                                      
      idx_data_sub.extend(list(np.arange(ix1,ix2)))                                                                                    
                                                                                                                                       
    ds_data_sub = ds_data.isel(data=idx_data_sub, drop=True)                                                                             
                                                                                                                                       
    # merge the hru and seg dimension variables with the result                                                                          
    return xr.merge([ds_poly_sub, ds_data_sub])

In [74]:
def remap(ds_ctsm, var_name, ds_mapping, ids=np.array([])):
                                                                                                                     
        # Get data from spatial weights file                                                                                                                                                                                
        wgtVals = ds_mapping['weight'].values                                                                                             
        i_index = ds_mapping['i_index'].values.astype(int)     
        j_index = ds_mapping['j_index'].values.astype(int)        # j_index --- LAT direction 
        allOutPolyIDs = ds_mapping['polyid'].values
        overlaps = ds_mapping['overlaps'].values.astype(int)
                                                                                                                                                                                                                                                          
        # read times variables                                                                                        
        times = {'time':      ds_ctsm['time']}                                                                                               
        nTimeSteps = len(times['time'])                                                  
                                                                                                                                        
        # set to zero based index and flip the N-S index (for read-in data array 0,0 is SW-corner)                                      
        j_index = j_index - 1              # j_index starts at North, and at 1 ... make 0-based                                         
        i_index = i_index - 1              # i_index starts at West, and a 1 ... make 0-based                                           
                                                                                                                                                                                                                                                              
        # check run end limits and calculate some useful dimensions                                                                     
        nOutPolys = len(allOutPolyIDs)                                                                                                                                                                    
        maxOverlaps = overlaps.max()                                                                                                    
                                                                                                                                        
        # calculate indices for subsetting polygon range (for all data)                                                                 
        overlapEndNdx   = np.cumsum(overlaps)                                                                                        
        overlapStartNdx = np.cumsum(overlaps)-overlaps                                                                                
                                                                                                                                        
        # assign weights and indices to a regular array (nOutPolys x maxOverlaps)                                                       
        matWgts = np.zeros((nOutPolys, maxOverlaps))                                                                                    
        for p in range(0, nOutPolys):                                                                                                   
            wgt_array = wgtVals[overlapStartNdx[p]:overlapEndNdx[p]]                                                                    
            wgt_sum = np.sum(wgt_array)                                                                                                 
            if wgt_sum>0:                                                                                                               
                matWgts[p, 0:overlaps[p]] = wgt_array/wgt_sum                                                                           
                                                                                                                                        
        # now use these weights in the avg val computation
        # now read the input timeseries file                                                                    
        dataVals = ds_ctsm[var_name].fillna(0.0).values # convert NaN to zero                                                                                                                           
                                                                                                                                        
        matDataVals = np.zeros((nOutPolys, maxOverlaps))                                                                                
        wgtedVals   = np.zeros((nTimeSteps, nOutPolys))                                                                       
                                                                                                                                        
        # format data into shape matching weights [nOutPolys, maxOverlaps]                                                              
        for t in range(0, nTimeSteps):                                                                                                  
                                                                                                                                        
            # reformat var data into regular matrix matching weights format (nOutPolygons, maxOverlaps)                                 
            #   used advanced indexing to extract matching input grid indices                                                           
            for p in range(0, nOutPolys):                                                                                               
                matDataVals[p, 0:overlaps[p]] = dataVals[t, [j_index[overlapStartNdx[p]:overlapEndNdx[p]]],[i_index[overlapStartNdx[p]:overlapEndNdx[p]]]]
                
            wgtedVals[t, ] = np.sum(matDataVals * matWgts, axis=1)   # produces vector of weighted values                               
                                                                                                                                        
        print(" averaged var %s over %d polygons for %d timesteps" % (var_name, nOutPolys, nTimeSteps))               
        return wgtedVals

In [4]:
# network variable required
network_vars = ["seg_id","upAll_start","upAll_count","allUpSegIndices"]

In [6]:
ds_network = xr.open_dataset("/glade/work/mizukami/test_mizuRoute/MERIT_basin_nldas/ancillary_data/ntopo_MERIT_Hydro_v1_aug.nc")[network_vars]
ds_network = ds_network.assign_coords(seg=ds_network.seg_id).load()
ds_mapping = xr.open_dataset("/glade/work/mizukami/py_mapping/MERIT_Hydro/mapping_v1/spatialweights_nldas0.125_to_merit_nldas.nc").load()
ds_lnd_hist = xr.open_mfdataset("/glade/derecho/scratch/mozhgana/archive/control/lnd/hist/control.clm2.h0a.1981-04.nc").load()

In [50]:
seg_id = 77006923

In [63]:
ix_start = ds_network.sel(seg=seg_id)["upAll_start"].values.astype("int") -1
ix_end = ix_start+ds_network.sel(seg=seg_id)["upAll_count"].values.astype("int")
upseg_ix_list = ds_network["allUpSegIndices"].isel(upAll=slice(ix_start,ix_end)).values.astype("int") -1
upseg_id_list = ds_network["seg_id"].isel(seg=upseg_ix_list)
print(upseg_id_list.values)

[77006923. 77007019. 77007140. 77007147. 77007461. 77007641. 77007925.
 77008204. 77009158. 77011916. 77011917. 77012172. 77011860. 77011749.
 77008335. 77008423. 77011804. 77011896. 77011650. 77011678. 77008987.
 77011356. 77011476. 77007475. 77007813. 77008075. 77011744. 77011881.
 77011645. 77011621. 77007184. 77007363. 77007731. 77007768. 77008078.
 77008139. 77008447. 77010871. 77010916. 77011140. 77010922. 77008499.
 77008767. 77009253. 77011139. 77011147. 77011237. 77011331. 77008685.
 77011123. 77011214. 77011398. 77008985. 77011303. 77011309.]


In [78]:
%%time
ds_mapping_sub = subset_mapping(ds_mapping, upseg_id_list)
var_upstream = np.mean(remap(ds_lnd_hist, "QRUNOFF", ds_mapping_sub))

 averaged var QRUNOFF over 55 polygons for 1 timesteps
CPU times: user 18.9 ms, sys: 0 ns, total: 18.9 ms
Wall time: 26.4 ms


In [79]:
var_upstream

1.3173881484031944e-05